# 📊 Sổ Tay 01: Khám Phá Dữ Liệu (EDA) & Tiền Xử Lý (Chia Tập 8:1:1)
**Đồ án**: Mạng nơ-ron đa lớp (MLP) Hồi quy dự đoán giá nhà (Lab 4)
**Phụ trách**: Bạn Huyền và Bạn My (Task 1)

Mục tiêu chính:
1. Nạp dữ liệu an toàn từ `data/raw/housing.csv` (20,640 mẫu).
2. Xử lý giá trị khuyết thiếu (NaN) trên cột `total_bedrooms`.
3. Khám phá phân bố biến mục tiêu giá nhà (`median_house_value`) và tương quan giữa các đặc trưng.
4. Phân chia tập dữ liệu chuẩn **8:1:1** (80% Train, 10% Validation, 10% Test) có phân tầng (`pd.qcut`).
5. Chuẩn hóa đặc trưng bằng `StandardScaler` đảm bảo **phòng chống rò rỉ dữ liệu (Data Leakage)**.


In [ ]:
# ==============================================================================
# BƯỚC 1: IMPORT CÁC THƯ VIỆN CẦN THIẾT
# ==============================================================================
# - Import các thư viện phân tích & trực quan hóa:
#     import pandas as pd
#     import numpy as np
#     import matplotlib.pyplot as plt
#     import seaborn as sns
# - Import các hàm hỗ trợ từ scikit-learn:
#     from sklearn.model_selection import train_test_split
#     from sklearn.preprocessing import StandardScaler
# - Import cấu hình và các module nội bộ của dự án:
#     import sys
#     sys.path.append('..')
#     from src.config import config
#     from src.data.loader import load_boston_housing_data, fetch_dataset_info
#     from src.data.preprocessing import handle_missing_values, split_data_8_1_1, scale_features
#     from src.features.feature_engineering import BostonFeatureEngineer
# - Cấu hình hiển thị biểu đồ: plt.style.use(config.STYLE)


## 1. Nạp Dữ Liệu & Kiểm Tra Cấu Trúc Tổng Quan (Data Inspection)


In [ ]:
# ==============================================================================
# BƯỚC 2: NẠP DỮ LIỆU TỪ RAW FILE VÀ KIỂM TRA THÔNG TIN SCHEMA
# ==============================================================================
# - Đọc file dữ liệu thô:
#     df, target_col = load_boston_housing_data(filepath=config.RAW_CSV_PATH)
# - In 5 dòng đầu tiên: df.head()
# - Kiểm tra thông tin cấu trúc: df.info()
# - Kiểm tra kích thước (Shape): mong đợi (20640, 10)
# - Thống kê mô tả các biến số: df.describe().T
# - Kiểm tra số lượng ô khuyết thiếu (Missing values):
#     missing_counts = df.isna().sum()
#     missing_counts[missing_counts > 0]  # total_bedrooms có ~207 giá trị NaN


## 2. Xử Lý Giá Trị Khuyết Thiếu (Missing Value Imputation)


In [ ]:
# ==============================================================================
# BƯỚC 3: ĐIỀN KHUYẾT THIẾU BẰNG PHƯƠNG PHÁP TRUNG VỊ (MEDIAN)
# ==============================================================================
# - Gọi hàm xử lý khuyết thiếu:
#     clean_df = handle_missing_values(df, strategy='median')
# - Kiểm tra lại để xác nhận 100% không còn ô khuyết:
#     assert clean_df.isna().sum().sum() == 0, "Dữ liệu vẫn còn giá trị NaN!"
# - Xác nhận kích thước dữ liệu sau khi làm sạch: clean_df.shape


## 3. Phân Tích Khám Phá EDA: Phân Bố Biến Mục Tiêu & Biến Đổi Log


In [ ]:
# ==============================================================================
# BƯỚC 4: VẼ BIỂU ĐỒ PHÂN BỐ GIÁ NHÀ (TARGET DISTRIBUTION)
# ==============================================================================
# - Tạo figure 1 hàng 2 cột:
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# - Cột 1: Phân bố giá gốc:
#     sns.histplot(clean_df[target_col], kde=True, ax=ax1, color='royalblue')
#     ax1.set_title('Phân Bố Giá Nhà Gốc (Lệch Phải & Cắt Ngưỡng ở 500k)')
# - Cột 2: Phân bố sau khi lấy Log1p:
#     sns.histplot(np.log1p(clean_df[target_col]), kde=True, ax=ax2, color='forestgreen')
#     ax2.set_title('Phân Bố Giá Nhà Sau Khi Biến Đổi Log1p (Gần Phân Bố Chuẩn)')
# - plt.tight_layout(); plt.show()


## 4. Ma Trận Hệ Số Tương Quan Pearson (Correlation Analysis)


In [ ]:
# ==============================================================================
# BƯỚC 5: TÍNH TOÁN MA TRẬN TƯƠNG QUAN VÀ VẼ HEATMAP
# ==============================================================================
# - Lọc các cột kiểu số để tính hệ số tương quan Pearson:
#     numeric_df = clean_df.select_dtypes(include=[np.number])
#     corr_matrix = numeric_df.corr()
# - Sắp xếp các đặc trưng theo mức độ tương quan với giá nhà:
#     corr_with_target = corr_matrix[target_col].sort_values(ascending=False)
#     print("Hệ số tương quan với biến mục tiêu:
", corr_with_target)
# - Vẽ biểu đồ nhiệt Heatmap:
#     plt.figure(figsize=(10, 8))
#     sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
#     plt.title('Ma Trận Tương Quan Pearson Giữa Các Đặc Trưng')
#     plt.show()
# - Nhận xét: Cột median_income có tương quan dương mạnh nhất với giá nhà (r xấp xỉ 0.69).


## 5. Phân Chia Tập Dữ Liệu Theo Tỷ Lệ Chuẩn 8:1:1 (Train - Val - Test)


In [ ]:
# ==============================================================================
# BƯỚC 6: CHIA TẬP THEO TỶ LỆ 8:1:1 TUÂN THỦ CODE MẪU PHASE 03
# ==============================================================================
# - Tách ma trận đặc trưng X và vector mục tiêu y:
#     X = clean_df.drop(columns=[target_col])
#     y = clean_df[target_col].copy()
# - Gọi hàm split_data_8_1_1 có phân tầng theo phân vị giá nhà:
#     X_train, X_val, X_test, y_train, y_val, y_test = split_data_8_1_1(
#         X, y, random_state=config.RANDOM_STATE, stratify=True
#     )
# - Kiểm tra số lượng dòng của từng tập:
#     print(f"Tổng số mẫu:     {len(clean_df):,}")
#     print(f"Train (80%):      {len(X_train):,} ({len(X_train)/len(clean_df)*100:.1f}%)")
#     print(f"Validation (10%): {len(X_val):,} ({len(X_val)/len(clean_df)*100:.1f}%)")
#     print(f"Test (10%):       {len(X_test):,} ({len(X_test)/len(clean_df)*100:.1f}%)")
# - Đảm bảo: len(X_train) == 16512, len(X_val) == 2064, len(X_test) == 2064.


## 6. Kỹ Thuật Đặc Trưng & Chuẩn Hóa StandardScaler (Chống Data Leakage)


In [ ]:
# ==============================================================================
# BƯỚC 7: XỬ LÝ BIẾN PHÂN LOẠI & CHUẨN HÓA BẰNG STANDARDSCALER
# ==============================================================================
# - Xử lý cột phân loại (ocean_proximity) bằng One-Hot Encoding trên cả 3 tập:
#     X_train_encoded = pd.get_dummies(X_train, drop_first=True)
#     X_val_encoded   = pd.get_dummies(X_val, drop_first=True).reindex(columns=X_train_encoded.columns, fill_value=0)
#     X_test_encoded  = pd.get_dummies(X_test, drop_first=True).reindex(columns=X_train_encoded.columns, fill_value=0)
# - Chuẩn hóa StandardScaler:
#     scaler = StandardScaler()
#     # BẮT BUỘC: fit_transform độc quyền trên Train, chỉ transform trên Val và Test
#     X_train_scaled = scaler.fit_transform(X_train_encoded)
#     X_val_scaled   = scaler.transform(X_val_encoded)
#     X_test_scaled  = scaler.transform(X_test_encoded)
# - Lưu bộ scaler và các tập dữ liệu đã chuẩn hóa vào results/saved_models/ và data/processed/.
# - Kiểm tra mean ~ 0 và std ~ 1 trên tập Train để xác nhận chuẩn hóa thành công!
